<a href="https://colab.research.google.com/github/basmah1111/-/blob/main/03_advanced_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 01.1 - Advanced Libraries Installation & Setup
# ==============================================================================
# تثبيت مكتبات الـ Advanced:
# - pytorch-crf: لتطبيق طبقات Conditional Random Fields
# - seqeval: لحساب مقاييس التقييم الدقيقة (F1-score, Precision, Recall) على مستوى الكلمات
# - torch: لإنشاء النماذج المخصصة و Custom Losses
!pip install pytorch-crf seqeval accelerate -q

import torch
import torch.nn as nn
from torchcrf import CRF
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

print(f"⚡ PyTorch Version: {torch.__version__}")
print(f"🚀 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# 02 - Complete & Self-Contained Dataset Processing & Loader Pipeline
# ==============================================================================
import os
from pathlib import Path
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, DataCollatorForTokenClassification

DATA_PATH = "processed_data/pa_tokenized"
MODEL_NAME = "UBC-NLP/MARBERTv2"

# 1️⃣ تجهيز الـ Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2️⃣ فحص وجود البيانات المعالجة سابقاً
def is_valid_dataset_dir(path):
    p = Path(path)
    return p.exists() and p.is_dir() and any(p.glob("*.json"))

if is_valid_dataset_dir(DATA_PATH):
    print(f"✅ تم العثور على البيانات المعالجة في: {DATA_PATH}")
    tokenized_pa = load_from_disk(DATA_PATH)
    print("✅ تم تحميل البيانات بنجاح!")
else:
    print(f"⚠️ المسار '{DATA_PATH}' غير موجود.")
    print("📥 جاري تحميل بيانات AraSeg 2026 الأصلية وتجهيزها...")

    # تحميل مجموعة البيانات الصحيحة الخاصة بالمسابقة
    pa = load_dataset("MBZUAI/AraSeg-2026-Shared-Task-PA")

    # دالة التقطيع والمحاذاة المطابقة تماماً لكود البريبروسسينج الخاص بك
    def tokenize_and_align_labels(examples):
        tokenized_inputs = tokenizer(
            examples["tokens"],
            is_split_into_words=True,
            truncation=True,
            max_length=512,
        )

        all_labels = []
        for i, labels in enumerate(examples["labels"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            previous_word_idx = None
            label_ids = []

            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(labels[word_idx])
                else:
                    label_ids.append(-100)
                previous_word_idx = word_idx
            all_labels.append(label_ids)

        tokenized_inputs["labels"] = all_labels
        return tokenized_inputs

    print("🔄 جاري تطبيق tokenize_and_align_labels على البيانات...")
    tokenized_pa = pa.map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=pa["train"].column_names
    )

    # حفظ البيانات المجهزة لضمان عدم تكرار المعالجة
    os.makedirs(DATA_PATH, exist_ok=True)
    tokenized_pa.save_to_disk(DATA_PATH)
    print(f"💾 تم حفظ البيانات بنجاح في: {DATA_PATH}")

print("\n📊 ملخص مجموعة البيانات المعالجة:")
print(tokenized_pa)

# 3️⃣ تجهيز Data Collator
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True
)

print("\n✅ تم تجهيز DataCollatorForTokenClassification بنجاح!")

In [ ]:
# ==============================================================================
# 03 - Advanced Metrics Setup (seqeval for Token Classification)
# ==============================================================================
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

# 1️⃣ قائمة الوسوم (Labels) المستخرجة من البيانات
# ملاحظة: استبدلي هذه القائمة بـ id2label المعتمد لديك في البيانات إذا كان متوفراً
label_list = ["O", "B-SEG", "I-SEG"]  # الوسوم الخاصة بمسابقة التقطيع AraSeg

def compute_metrics(p):
    predictions, labels = p
    # تحويل التوقعات إلى الأحتمالية الأعلى (Argmax)
    predictions = np.argmax(predictions, axis=2)

    # استبعاد التوكنات التي تحمل -100 وتحويل الأرقام إلى الوسوم النصية الأصلية
    true_predictions = [
        [label_list[p_i] for (p_i, l_i) in zip(prediction, label) if l_i != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l_i] for (p_i, l_i) in zip(prediction, label) if l_i != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # حساب المقاييس المعيارية
    results = {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

    return results

print("✅ تم تعريف دالة compute_metrics المتقدمة بنجاح!")

In [ ]:
# ==============================================================================
# 04 - Advanced Model Architecture (MARBERTv2 + CRF Layer)
# ==============================================================================
import torch
import torch.nn as nn
from transformers import AutoModel, PreTrainedModel, AutoConfig
from torchcrf import CRF

class MARBERTForTokenClassificationWithCRF(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.num_labels = num_labels

        # 1️⃣ تحميل نموذج MARBERTv2 الأساسي
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        # 2️⃣ طبقة Dropout لمنع الـ Overfitting
        self.dropout = nn.Dropout(0.2)

        # 3️⃣ طبقة Classifier للتحويل لوسوم الأبعاد (Num Labels)
        self.classifier = nn.Linear(hidden_size, num_labels)

        # 4️⃣ طبقة CRF المتقدمة لحساب التسلسلات الذكية
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask=None, labels=None):
        # استخراج التمثيلات من MARBERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs[0]
        sequence_output = self.dropout(sequence_output)

        # تحويل النتائج إلى Probabilities لكل Label
        emissions = self.classifier(sequence_output)

        if labels is not None:
            # أثناء التدريب: نحسب الـ Loss باستخدام الـ CRF
            # تحويل القيم -100 إلى 0 مؤقتاً لأن CRF لا يقبل القيم السالبة في القناع
            mask = (attention_mask == 1)
            clean_labels = labels.clone()
            clean_labels[clean_labels == -100] = 0

            # CRF Loss يعيد log-likelihood، نضربه بـ -1 لنحصل على loss إيجابي للتدريب
            log_likelihood = self.crf(emissions, clean_labels, mask=mask, reduction='mean')
            loss = -log_likelihood
            return {"loss": loss, "logits": emissions}
        else:
            # أثناء الـ Inference / Test: فك التشفير واستخراج أفضل تسلسل
            mask = (attention_mask == 1)
            best_tags = self.crf.decode(emissions, mask=mask)
            return {"predictions": best_tags, "logits": emissions}

# 🚀 بناء نسخة من الموديل المتقدم
MODEL_NAME = "UBC-NLP/MARBERTv2"
NUM_LABELS = 3  # (O, B-SEG, I-SEG)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MARBERTForTokenClassificationWithCRF(MODEL_NAME, num_labels=NUM_LABELS)
model.to(device)

print(f"✅ تم بناء نموذج MARBERTv2 + CRF بنجاح ونقله إلى: {device}")

In [ ]:
# ==============================================================================
# 05 - Fixed Advanced Custom Training Loop (MARBERTv2 + CRF)
# ==============================================================================
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from seqeval.metrics import f1_score, precision_score, recall_score

# 1️⃣ دالة التقييم المعدلة لتقبل المخرجات المباشرة من CRF
label_list = ["O", "B-SEG", "I-SEG"]

def compute_metrics_crf(all_preds, all_labels):
    # تحويل الأرقام التسلسلية لأسماء الوسوم النصية مع تجاهل -100
    true_predictions = [
        [label_list[p] for (p, l) in zip(pred_seq, label_seq) if l != -100]
        for pred_seq, label_seq in zip(all_preds, all_labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred_seq, label_seq) if l != -100]
        for pred_seq, label_seq in zip(all_preds, all_labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

# 2️⃣ إعداد الـ DataLoaders
BATCH_SIZE = 16
train_dataloader = DataLoader(
    tokenized_pa["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_pa["validation"] if "validation" in tokenized_pa else tokenized_pa["test"],
    batch_size=BATCH_SIZE,
    collate_fn=data_collator
)

# 3️⃣ تحديد الـ Hyperparameters والـ Optimizer
EPOCHS = 3
LEARNING_RATE = 3e-5

optimizer = torch.optim.AdamW([
    {'params': model.bert.parameters(), 'lr': LEARNING_RATE},
    {'params': model.classifier.parameters(), 'lr': LEARNING_RATE * 10},
    {'params': model.crf.parameters(), 'lr': LEARNING_RATE * 10}
], weight_decay=0.01)

total_steps = len(train_dataloader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

# 4️⃣ حلقة التدريب والتقييم (Training & Validation Loop)
print(f"🚀 بدء تدريب النموذج المتقدم على {device} لمُدة {EPOCHS} Epochs...\n")

for epoch in range(EPOCHS):
    # --- أ) مرحلة التدريب (Training Phase) ---
    model.train()
    total_train_loss = 0

    train_pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Training]")
    for batch in train_pbar:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs["loss"]

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()
        train_pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"✨ Epoch {epoch+1} - Average Training Loss: {avg_train_loss:.4f}")

    # --- ب) مرحلة التقييم (Validation Phase) ---
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(eval_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Evaluation]"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = outputs["predictions"] # قائمة جاهزة من CRF decode

            for i, label_seq in enumerate(labels.cpu().numpy()):
                all_preds.append(predictions[i])
                all_labels.append(label_seq)

    # حساب الـ Metrics باستخدام الدالة المخصصة لـ CRF
    eval_metrics = compute_metrics_crf(all_preds, all_labels)

    print(f"📊 Validation Results - Epoch {epoch+1}:")
    print(f"   - Precision : {eval_metrics['precision']:.4f}")
    print(f"   - Recall    : {eval_metrics['recall']:.4f}")
    print(f"   - F1-Score  : {eval_metrics['f1']:.4f}\n" + "-"*50)

# 💾 حفظ أوزان الموديل المتقدم
OUTPUT_MODEL_DIR = "marbert_crf_advanced.pt"
torch.save(model.state_dict(), OUTPUT_MODEL_DIR)
print(f"🎉 تم حفظ الموديل المتقدم بنجاح في: {OUTPUT_MODEL_DIR}")

In [ ]:
# ==============================================================================
# 06 - Test Inference & Submission Generation (MARBERTv2 + CRF)
# ==============================================================================
import pandas as pd
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# 1️⃣ تجهيز DataLoader لبيانات الـ Test
test_dataloader = DataLoader(
    tokenized_pa["test"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

# 2️⃣ تشغيل الموديل المتقدم على بيانات الـ Test
model.eval()
test_predictions = []

print("🚀 جاري معالجة بيانات الـ Test واستخراج التقطيعات المتوقعة...")

with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Predicting Test Set"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Forward pass واستخراج التوقعات عبر CRF Decode
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = outputs["predictions"]

        for pred_seq in predictions:
            # تحويل أرقام الوسوم إلى مسمياتها النصية (O, B-SEG, I-SEG)
            named_preds = [label_list[tag] for tag in pred_seq]
            test_predictions.append(named_preds)

# 3️⃣ حفظ التوقعات في ملف CSV جاهز للتسليم
SUBMISSION_PATH = "submission_advanced.csv"

submission_df = pd.DataFrame({
    "id": range(len(test_predictions)),
    "predictions": [" ".join(seq) for seq in test_predictions]
})

submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"\n🎉 تم توليد ملف التوقعات النهائي بنجاح وتصديره إلى: {SUBMISSION_PATH}")
print("\nعيّنة من النتائج الأولى:")
print(submission_df.head())

In [ ]:
# ==============================================================================
# 07 - Optimized CRF Training & Text-Only Comparison Table
# ==============================================================================
import pandas as pd
import numpy as np

# 1️⃣ طباعة الجدول النصي مباشرة للتقييم (بدون رسم بياني)
def print_comparison_table(baseline_f1, advanced_metrics):
    comparison_data = {
        "Model Architecture": ["Baseline (MARBERTv2)", "Advanced (MARBERTv2 + CRF)"],
        "Precision": [0.8920, round(advanced_metrics["precision"], 6)],
        "Recall":    [0.8850, round(advanced_metrics["recall"], 6)],
        "F1-Score":  [0.8885, round(advanced_metrics["f1"], 6)]
    }

    df_comparison = pd.DataFrame(comparison_data)

    print("\n📊 جدول مقارنة الأداء النهائي (Performance Comparison Table):")
    print("=" * 65)
    print(df_comparison.to_string(index=False))
    print("=" * 65)

# طباعة الجدول بالنتائج الحالية
print_comparison_table(0.8885, eval_metrics)

In [ ]:
# تعديل بسيط في Optimizer داخل خلية التدريب لتسريع تعلم الـ CRF:
optimizer = torch.optim.AdamW([
    {'params': model.bert.parameters(), 'lr': 2e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-3},
    {'params': model.crf.parameters(), 'lr': 1e-2}  # رفع Learning Rate للـ CRF لتقترب بسرعة
], weight_decay=0.01)